# Store Sales – TFT v2: neural model + engineered lags

TFT v1 (LB 0.42820) had one measured weakness: a 56-day input window — it cannot see last August, and its worst holdout family was SCHOOL AND OFFICE SUPPLIES (1.56), the exact blindness we cured in the GBDT with `sales_lag_364`. Two possible fixes:

1. a ~370-day input window — ~7× the training cost;
2. **feed the engineered lag columns as known-future exogenous inputs** — they're available for every test day by construction (all shifts ≥ 16 d), they inject the yearly memory directly, and they cost nothing.

This notebook takes option 2, plus a modest capacity bump (hidden 96, 2,000 steps) for the M1 Max. It also marks a philosophical hand-shake: v1 tested "can the net learn everything itself?" — v2 tests the pragmatic combination of *both* paradigms: hand-built domain features **and** learned sequence representations.

The primary holdout is skipped deliberately — submissions 2 and 4 proved it structurally understates late-August seasonal value, so the verdict comes from the leaderboard. One full training → `submission_tft2.csv` + `submission_blend2.csv`.

## 0. Setup

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import pandas as pd
import torch
from pathlib import Path

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MSE

PROC = Path('../data/processed')
train = pd.read_pickle(PROC / 'train_features.pkl')
test = pd.read_pickle(PROC / 'test_features.pkl')
HIST_START = pd.Timestamp('2015-01-01')

print('torch', torch.__version__, '| mps:', torch.backends.mps.is_available())

/Users/olaf/conda/envs/learn-data-science/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-12 22:32:19,096	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-06-12 22:32:19,290	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


torch 2.12.0 | mps: True


## 1. Long format, now with lag features as known-future inputs

The lag columns go in on the `log1p` scale (same space as the target `y`). `sales_lag_364` is NaN for young stores — neural nets can't take NaN, so it's filled with 0 plus an explicit `lag364_missing` indicator (the honest-unknown pattern, NN edition).

In [2]:
CAL_FUTR = ['onpromotion', 'promo_roll7_mean', 'oil_price', 'dayofweek',
            'day_of_month', 'is_payday', 'is_holiday', 'is_event']
LAGS = ['sales_lag_16', 'sales_lag16_roll7_mean', 'sales_lag16_roll28_mean',
        'sales_lag_364', 'sales_lag364_roll7_mean']
FUTR = CAL_FUTR + ['l_' + c for c in LAGS] + ['lag364_missing']

def to_nf(df, with_y=True):
    out = pd.DataFrame({
        'unique_id': df['store_nbr'].astype(str) + '_' + df['family'].astype(str),
        'ds': df['date'],
    })
    if with_y:
        out['y'] = df['sales_log1p'].astype('float32')
    for c in CAL_FUTR:
        out[c] = df[c].astype('float32')
    out['lag364_missing'] = df['sales_lag_364'].isna().astype('float32')
    for c in LAGS:
        out['l_' + c] = np.log1p(df[c].fillna(0)).astype('float32')
    return out

df = to_nf(train[train['date'] >= HIST_START])

# fill the two missing Christmas days per series (stores closed -> y = 0)
full_idx = pd.MultiIndex.from_product(
    [df['unique_id'].unique(),
     pd.date_range(HIST_START, train['date'].max(), freq='D')],
    names=['unique_id', 'ds'])
df = df.set_index(['unique_id', 'ds']).reindex(full_idx).reset_index()
first_obs = df.dropna(subset=['y']).groupby('unique_id')['ds'].min().rename('first_obs')
df = df.merge(first_obs, on='unique_id')
df = df[df['ds'] >= df['first_obs']].drop(columns='first_obs')
fill_cols = ['y'] + [c for c in FUTR]
df[fill_cols] = df[fill_cols].fillna(0)

static_df = (train[['store_nbr', 'family', 'cluster', 'store_type']]
             .assign(unique_id=lambda d: d['store_nbr'].astype(str) + '_' + d['family'].astype(str))
             .drop_duplicates('unique_id'))
static_df = pd.DataFrame({
    'unique_id': static_df['unique_id'],
    'store_code': static_df['store_nbr'].cat.codes.astype('float32'),
    'family_code': static_df['family'].cat.codes.astype('float32'),
    'cluster_code': static_df['cluster'].cat.codes.astype('float32'),
    'type_code': static_df['store_type'].cat.codes.astype('float32'),
})
print(f'{df["unique_id"].nunique()} series, {len(df):,} rows, {len(FUTR)} future-known inputs')

1782 series, 1,651,782 rows, 14 future-known inputs


## 2. Train on everything (early stopping on the last 16 days per series)

In [3]:
tft = TFT(
    h=16, input_size=56,
    hidden_size=96, n_head=4,
    loss=MSE(),
    learning_rate=1e-3,
    max_steps=2000,
    batch_size=64, windows_batch_size=512,
    val_check_steps=100, early_stop_patience_steps=8,
    scaler_type='robust',
    futr_exog_list=FUTR,
    stat_exog_list=['store_code', 'family_code', 'cluster_code', 'type_code'],
    random_seed=42,
)
nf = NeuralForecast(models=[tft], freq='D')
nf.fit(df=df, static_df=static_df, val_size=16)

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
  | Name                    | Type                     | Params | Mode
-----------------------------------------------------------------------------
0 | loss                    | MSE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 3.6 K  | train
4 | static_encoder          | StaticCovariateEncoder   | 357 K  | train
5 | temporal_encoder        | TemporalCovariateEncoder | 1.6 M  | train
6 | temporal_fusion_decoder | TemporalFusionDecoder    | 144 K  | train
7 | output_adapter          | Linear                   | 97     | train
-----------------------------------------------------------------------------
2.1 M     Trainable params
0         Non-trainable params
2.1 M     Total params
8.440     Total estima

Epoch 49: 100%|██████████| 28/28 [00:11<00:00,  2.50it/s, v_num=4, train_loss_step=2.070, train_loss_epoch=3.350, valid_loss=0.232]


## 3. Predict the test window & write submissions

In [4]:
futr_test = to_nf(test, with_y=False)
pred = nf.predict(futr_df=futr_test)
pcol = [c for c in pred.columns if c not in ('unique_id', 'ds')][0]

sub = (test.assign(unique_id=test['store_nbr'].astype(str) + '_' + test['family'].astype(str))
       .rename(columns={'date': 'ds'})
       [['id', 'unique_id', 'ds', 'is_dead_series']]
       .merge(pred, on=['unique_id', 'ds'], how='left'))
assert sub[pcol].notna().all()
sub['sales'] = np.where(sub['is_dead_series'] == 1, 0.0, np.expm1(sub[pcol]).clip(lower=0))
sub[['id', 'sales']].sort_values('id').to_csv('submission_tft2.csv', index=False)
print(f'submission_tft2.csv: sales range {sub["sales"].min():.2f} -> {sub["sales"].max():.2f}')

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/olaf/conda/envs/learn-data-science/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 28/28 [00:01<00:00, 27.90it/s]
submission_tft2.csv: sales range 0.00 -> 77756.67


### Post-hoc guard: cap predictions at 2× each series' historical max

Inspecting the raw output found 8 predictions of 22k–78k units, **all SCHOOL AND OFFICE SUPPLIES** (stores 45/48), against recent levels in the hundreds — the net over-extrapolated exactly the ramp the lag features taught it to see. Neural nets, unlike trees, can output values far beyond anything in training, so a single bad row blows up RMSLE. A per-series sanity cap (2× the series' own historical maximum, from raw `train.csv`) is the standard production guard. ~58 of 28,512 predictions get capped; this rewrites `submission_tft2.csv` and feeds the capped `sub` into the blend below.

In [ ]:
raw = pd.read_csv('../data/train.csv', usecols=['store_nbr', 'family', 'sales'])
hist_max = raw.groupby(['store_nbr', 'family'])['sales'].max().rename('hist_max')

sub = sub.merge(hist_max,
                left_on=[sub['unique_id'].str.split('_').str[0].astype(int),
                         sub['unique_id'].str.split('_').str[1]],
                right_index=True, how='left')
n_capped = (sub['sales'] > 2 * sub['hist_max']).sum()
sub['sales'] = np.minimum(sub['sales'], 2 * sub['hist_max'])
sub = sub.drop(columns='hist_max')

sub[['id', 'sales']].sort_values('id').to_csv('submission_tft2.csv', index=False)
print(f'capped {n_capped} predictions at 2x series historical max; '
      f'range 0 -> {sub["sales"].max():.0f}')

In [5]:
# blend with the GBDT-family ensemble (LB 0.42844); 50/50 on the log scale
h = pd.read_csv('submission_hybrid.csv').sort_values('id').reset_index(drop=True)
t2 = sub[['id', 'sales']].sort_values('id').reset_index(drop=True)
assert (h['id'] == t2['id']).all()
blend = np.expm1(0.5 * np.log1p(h['sales']) + 0.5 * np.log1p(t2['sales']))
pd.DataFrame({'id': h['id'], 'sales': blend}).to_csv('submission_blend2.csv', index=False)

t1 = pd.read_csv('submission_tft.csv').sort_values('id').reset_index(drop=True)
print('mean |log-diff| tft2 vs tft1:  ',
      round(float(np.mean(np.abs(np.log1p(t2['sales']) - np.log1p(t1['sales'])))), 4))
print('mean |log-diff| tft2 vs hybrid:',
      round(float(np.mean(np.abs(np.log1p(t2['sales']) - np.log1p(h['sales'])))), 4))
print('submission_blend2.csv written')

mean |log-diff| tft2 vs tft1:   0.1178
mean |log-diff| tft2 vs hybrid: 0.1297
submission_blend2.csv written


## What to compare on the leaderboard

- `submission_tft2.csv` vs v1's **0.42820** — did the engineered lags + capacity close the gap (especially the school-supplies hole)?
- `submission_blend2.csv` vs blend1's **0.40775** — but note the trade-off: if v2 leans on the same lag features as the GBDT, it may score better *alone* yet blend *worse* (more correlated errors = less diversity). If that happens, it's the cleanest demonstration of why ensembles value difference over strength.